# Carrot tuning system: initial model

This notebook is a **high-level, executable mock** of the system we are building. It does not call Azure, Foundry, an LLM, or Kaggriculture. It does not edit policy source.

Its purpose is to make the intended control flow concrete before we build the durable components: a deterministic inner experiment loop, an evidence packet, an outer evals/reasoning loop, and a human approval boundary.

The completed carrot experiments are used only as illustrative input data. The notebook’s evaluator is intentionally mocked.

## 1. The idea in one picture

```mermaid
flowchart TD
    approval["Human approves a bounded experiment specification"]
    inner["Inner loop (local and deterministic)"]
    evidence["baseline → checkpoint replay/suffix → clean audit → evidence packet"]
    outer["Outer loop (evals + reasoning)"]
    proposal["validate evidence → explain result → propose next bounded test"]
    review["Human approves the next envelope or applies a reviewed source change"]

    approval --> inner --> evidence --> outer --> proposal --> review
    review --> approval
```

Fallback for notebook viewers without Mermaid rendering:

```text
Human approves a bounded experiment specification
                 |
                 v
Inner loop (local and deterministic)
  baseline -> checkpoint replay/suffix -> clean audit -> evidence packet
                 |
                 v
Outer loop (evals + reasoning)
  validate evidence -> explain result -> propose next bounded test
                 |
                 v
Human approves the next envelope or applies a reviewed source change
```

The evaluator is the score authority because it runs Kaggriculture locally. The outer loop is an interpretation and experiment-design authority, not a source-editing authority.

## 2. Guardrails encoded as data

A useful first design choice is to make the approved envelope explicit. The system can automatically run values inside this specification, but it cannot silently add values, change the target source, or promote a value into policy.

In [17]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Literal


# Frozen records make an approved instruction immutable once it is handed to the runner.
# A real runner still validates the serialized v1 schema before it trusts this model.
@dataclass(frozen=True)
class Suite:
    """Matched controls for every candidate in one experiment."""

    # These fields pin the comparison controls; candidates cannot choose a new opponent
    # or a friendlier seed/seat/turn suite after seeing an earlier result.
    # These fields are immutable once the suite is created.
    opponent_id: str
    seeds: tuple[int, ...]
    seats: tuple[int, ...] = (0, 1)
    steps: int = 720


@dataclass(frozen=True)
class ExperimentSpec:
    """A human-approved, bounded inner-loop instruction."""

    # Identity and scope make this one named, reviewable experiment—not an open-ended search.
    experiment_id: str
    decision_graph_id: str

    # The runner receives one named in-memory parameter override; this model has no
    # source path or source-writing operation, so it cannot authorize a policy edit.
    parameter_name: str
    baseline_value: float

    # A tuple is an explicit, fixed envelope. The runner may evaluate these values,
    # but must reject any attempt to append a value after results are observed.
    candidate_values: tuple[float, ...]
    suite: Suite
    purpose: str

    # The default makes independent clean audits mandatory for every candidate.
    requires_clean_audit: bool = True


# This is the whole approved envelope for the illustrative coarse bracket.
# It deliberately names every candidate rather than deriving a new value from results.
coarse_seed_bracket = ExperimentSpec(
    experiment_id="carrot-seed-opportunity-coarse-v1",
    decision_graph_id="carrot_investment",
    parameter_name="seed_opportunity_value",
    baseline_value=20.0,
    candidate_values=(10.0, 40.0, 60.0, 80.0),
    suite=Suite(
        opponent_id="target.agents.carrot.shared.carrot_conveyor_agent",
        seeds=tuple(range(42, 72)),
    ),
    purpose="Measure the direction and threshold behavior of preserving a carrot seed.",
)

# The next cell presents this envelope without exposing the implementation details.

In [18]:
from IPython.display import Markdown, display


def render_experiment_envelope(spec: ExperimentSpec) -> None:
    """Presentation only: clearing this output never changes the specification."""
    candidates = ", ".join(f"`{value:g}`" for value in spec.candidate_values)
    display(Markdown(
        f"""### Approved experiment envelope

| Field | Approved value |
|:---|:---|
| Experiment | `{spec.experiment_id}` |
| Decision graph | `{spec.decision_graph_id}` |
| In-memory override | `{spec.parameter_name}`; baseline `{spec.baseline_value:g}` |
| Candidate envelope | {candidates} |
| Matched suite | Conveyor opponent; seeds {spec.suite.seeds[0]}–{spec.suite.seeds[-1]}; both seats; {spec.suite.steps} turns |
| Verification | Clean audit required for every candidate |

> This cell describes a bounded instruction. It authorizes no source edit or new candidate value.
"""
    ))


render_experiment_envelope(coarse_seed_bracket)

### Approved experiment envelope

| Field | Approved value |
|:---|:---|
| Experiment | `carrot-seed-opportunity-coarse-v1` |
| Decision graph | `carrot_investment` |
| In-memory override | `seed_opportunity_value`; baseline `20` |
| Candidate envelope | `10`, `40`, `60`, `80` |
| Matched suite | Conveyor opponent; seeds 42–71; both seats; 720 turns |
| Verification | Clean audit required for every candidate |

> This cell describes a bounded instruction. It authorizes no source edit or new candidate value.


## 3. Inner-loop output: an evidence packet

The real inner loop will invoke the local checkpoint/resume evaluator. A candidate follows baseline observations until its first different action; it reuses the baseline when there is no divergence, otherwise restores the exact pre-decision state and simulates the suffix. A clean full rerun audits every result.

For this notebook, the mock evaluator returns a frozen snapshot of the completed coarse `seed_opportunity_value` run. This is deliberately not a simulation substitute. It only demonstrates the data contract the real evaluator should produce.

In [19]:
# Each row is an immutable fact about one declared candidate, not a recommendation.
@dataclass(frozen=True)
class CandidateEvidence:
    # This is the exact value from the approved envelope that was evaluated.
    value: float
    # Paired statistics compare the same seed and seat with the baseline; they keep
    # an apparent improvement from being confused with a different game draw.
    mean_reward: float
    mean_paired_delta: float
    # Divergence says how often replay had to restore state and simulate a suffix.
    diverged_matches: int
    total_matches: int
    # An audit count and explicit status make verification visible to the outer loop.
    # A real evaluator must report a failed audit; it may not silently replace it.
    clean_audits_passed: int
    audit_status: Literal["passed", "failed"]


# The packet freezes the evaluator's evidence before the reasoning layer sees it.
@dataclass(frozen=True)
class EvidencePacket:
    # Identity, target revision, and controls bind the conclusions to one reproducible run.
    experiment_id: str
    decision_graph_id: str
    target_revision: str
    suite: Suite
    baseline_mean_reward: float
    # Tuple ordering preserves the complete approved candidate set for later review.
    candidates: tuple[CandidateEvidence, ...]
    # This deterministic gate lets later code stop before interpreting unaudited evidence.
    evaluator_status: Literal["complete", "audit_failed"]
    notes: tuple[str, ...] = ()


# Frozen illustrative evidence from the already-completed local coarse screen.
# This fixture explains the packet shape only; it cannot score a new candidate.
# The real implementation will build this from JSON result files.
HISTORICAL_SEED_RESULTS = {
    10.0: (6469.02, 0.00, 0),
    40.0: (6416.65, -52.37, 24),
    60.0: (5631.28, -837.73, 48),
    80.0: (4845.73, -1623.28, 60),
}


def mock_inner_loop(spec: ExperimentSpec) -> EvidencePacket:
    """Mock only: returns fixture data and never runs a game."""
    # The expected count comes from the approved suite, so missing audits are detectable.
    expected_matches = len(spec.suite.seeds) * len(spec.suite.seats)
    candidates = tuple(
        CandidateEvidence(
            value=value,
            mean_reward=HISTORICAL_SEED_RESULTS[value][0],
            mean_paired_delta=HISTORICAL_SEED_RESULTS[value][1],
            diverged_matches=HISTORICAL_SEED_RESULTS[value][2],
            total_matches=expected_matches,
            clean_audits_passed=expected_matches,
            audit_status="passed",
        )
        # The mock iterates only values named by the specification; it never searches.
        for value in spec.candidate_values
    )
    return EvidencePacket(
        experiment_id=spec.experiment_id,
        decision_graph_id=spec.decision_graph_id,
        target_revision="6ace5fed9018fef45642b41ea11d0bd741a508a4",
        suite=spec.suite,
        baseline_mean_reward=6469.02,
        candidates=candidates,
        # A real evaluator would set audit_failed here if any clean rerun disagreed.
        evaluator_status="complete",
        notes=("Mock packet populated from completed local evidence.",),
    )


packet = mock_inner_loop(coarse_seed_bracket)
# The next cell presents this packet without exposing the implementation details.

In [20]:
from IPython.display import Markdown, display


def render_packet_overview(packet: EvidencePacket) -> None:
    """Presentation only: clearing this output never changes the evidence packet."""
    display(Markdown(
        f"""### Evidence packet

| Field | Recorded fact |
|:---|:---|
| Experiment | `{packet.experiment_id}` |
| Decision graph | `{packet.decision_graph_id}` |
| Target revision | `{packet.target_revision}` |
| Baseline mean reward | {packet.baseline_mean_reward:,.2f} |
| Candidate records | {len(packet.candidates)}, covering the complete approved envelope |
| Evaluator status | `{packet.evaluator_status}` |
| Provenance | {packet.notes[0]} |

> This packet is evidence for later deterministic evaluation; it does not recommend a policy change.
"""
    ))


render_packet_overview(packet)

### Evidence packet

| Field | Recorded fact |
|:---|:---|
| Experiment | `carrot-seed-opportunity-coarse-v1` |
| Decision graph | `carrot_investment` |
| Target revision | `6ace5fed9018fef45642b41ea11d0bd741a508a4` |
| Baseline mean reward | 6,469.02 |
| Candidate records | 4, covering the complete approved envelope |
| Evaluator status | `complete` |
| Provenance | Mock packet populated from completed local evidence. |

> This packet is evidence for later deterministic evaluation; it does not recommend a policy change.


## 4. Inspect the deterministic evidence before reasoning

This is the critical separation: the evaluation layer creates facts about the stated experiment; the reasoning layer is not allowed to reinterpret an audit failure into success.

In [21]:
from IPython.display import Markdown, display


def render_evidence_summary(packet: EvidencePacket) -> None:
    """Presentation only: clearing this output never changes the evidence packet."""
    rows = "\n".join(
        f"| {row.value:g} | {row.mean_reward:,.2f} | {row.mean_paired_delta:+,.2f} "
        f"| {row.diverged_matches}/{row.total_matches} "
        f"| {row.clean_audits_passed}/{row.total_matches} | `{row.audit_status}` |"
        for row in packet.candidates
    )
    display(Markdown(
        f"""### Deterministic evidence summary

**Evaluator status:** `{packet.evaluator_status}`  
**Target revision:** `{packet.target_revision}`  
**Matched suite:** {len(packet.suite.seeds)} seeds × {len(packet.suite.seats)} seats = {len(packet.suite.seeds) * len(packet.suite.seats)} games per candidate

| Candidate | Mean reward | Paired delta | Divergence | Clean audits | Status |
|---:|---:|---:|---:|---:|:---|
{rows}
"""
    ))


render_evidence_summary(packet)

### Deterministic evidence summary

**Evaluator status:** `complete`  
**Target revision:** `6ace5fed9018fef45642b41ea11d0bd741a508a4`  
**Matched suite:** 30 seeds × 2 seats = 60 games per candidate

| Candidate | Mean reward | Paired delta | Divergence | Clean audits | Status |
|---:|---:|---:|---:|---:|:---|
| 10 | 6,469.02 | +0.00 | 0/60 | 60/60 | `passed` |
| 40 | 6,416.65 | -52.37 | 24/60 | 60/60 | `passed` |
| 60 | 5,631.28 | -837.73 | 48/60 | 60/60 | `passed` |
| 80 | 4,845.73 | -1,623.28 | 60/60 | 60/60 | `passed` |


## 5. Outer evals loop: classify evidence, do not make policy edits

The first outer-loop component can be deterministic too. It checks that audits passed, identifies whether a candidate was inert or harmful on this suite, and emits constraints for the reasoning step. A future Foundry evaluator can implement richer review, but it should preserve these non-negotiable checks.

In [22]:
# This immutable result separates deterministic classification from later reasoning.
@dataclass(frozen=True)
class EvaluationAssessment:
    # A failed audit makes the entire packet unusable, even if some rows look favorable.
    status: Literal["evidence_valid", "audit_failed"]
    # Findings explain the classification; they are not a source-change recommendation.
    findings: tuple[str, ...]
    # These values are ruled out for this suite; a new scenario still needs a fresh approved envelope.
    prohibited_next_values: tuple[float, ...]


def evaluate_packet(packet: EvidencePacket) -> EvaluationAssessment:
    # Completeness and clean audits are a gate: never reason from partial or untrusted evidence.
    if packet.evaluator_status != "complete" or any(
        row.audit_status != "passed" for row in packet.candidates
    ):
        return EvaluationAssessment(
            status="audit_failed",
            findings=("Do not interpret or extend this experiment until audits are repaired.",),
            prohibited_next_values=(),
        )

    # Zero divergences means this candidate never changed a decision on this specific suite.
    inert = [row.value for row in packet.candidates if row.diverged_matches == 0]
    # The paired delta compares each game with its matched baseline, so below zero is harm here.
    harmful = [row.value for row in packet.candidates if row.mean_paired_delta < 0]
    return EvaluationAssessment(
        status="evidence_valid",
        findings=(
            f"Inert values on this suite: {inert}.",
            f"Values with negative mean paired reward: {harmful}.",
            # This is deliberately a mock deterministic rule, not an automatic policy edit.
            "The current evidence does not support increasing seed opportunity above 20.",
        ),
        prohibited_next_values=tuple(harmful),
    )


# Apply the deterministic gate before any later reasoning component sees this packet.
assessment = evaluate_packet(packet)
# The next cell presents this assessment without exposing the implementation details.

In [23]:
from IPython.display import Markdown, display


def render_evaluation_assessment(
    packet: EvidencePacket, assessment: EvaluationAssessment
) -> None:
    """Presentation only: clearing this output never changes the assessment."""
    prohibited = (
        ", ".join(f"`{value:g}`" for value in assessment.prohibited_next_values)
        or "None"
    )
    audit_summary = (
        "all candidate clean audits passed"
        if all(row.audit_status == "passed" for row in packet.candidates)
        else "one or more candidate clean audits did not pass"
    )
    findings = "\n".join(f"- {finding}" for finding in assessment.findings)
    display(Markdown(
        f"""### Deterministic evaluation assessment

| Check | Result |
|:---|:---|
| Gate status | `{assessment.status}` |
| Evidence packet | `{packet.evaluator_status}`; {audit_summary} |
| Values ruled out for this conveyor suite | {prohibited} |

**Findings**

{findings}

> This deterministic gate constrains later reasoning. It neither chooses a new candidate nor edits policy source.
"""
    ))


render_evaluation_assessment(packet, assessment)

### Deterministic evaluation assessment

| Check | Result |
|:---|:---|
| Gate status | `evidence_valid` |
| Evidence packet | `complete`; all candidate clean audits passed |
| Values ruled out for this conveyor suite | `40`, `60`, `80` |

**Findings**

- Inert values on this suite: [10.0].
- Values with negative mean paired reward: [40.0, 60.0, 80.0].
- The current evidence does not support increasing seed opportunity above 20.

> This deterministic gate constrains later reasoning. It neither chooses a new candidate nor edits policy source.


## 6. Reasoning loop: propose the next bounded uncertainty to resolve

This mock represents the job of a future reasoning agent. It receives the evidence packet and evaluation assessment, explains a proposed next test, and returns a bounded specification. It cannot write Kaggriculture source or expand its own candidate envelope.

Here, the correct next question is not “try an even higher value.” The observed bracket already shows higher static seed opportunity is harmful for the conveyor suite. The useful next question is whether this value should be supplied by a competing decision graph in different farm contexts.

In [24]:
# This immutable record is a proposed bounded test, not authority to run or apply it.
@dataclass(frozen=True)
class ReasoningProposal:
    # The uncertainty the proposal intends to resolve.
    hypothesis: str
    # The evidence-based reason this test is more useful than extending the old bracket.
    explanation: str
    # A complete envelope that a human must still approve before any local execution.
    next_experiment: ExperimentSpec
    # This remains true even when the proposal itself is internally well formed.
    requires_human_approval: bool = True


def mock_reasoning_loop(packet: EvidencePacket, assessment: EvaluationAssessment) -> ReasoningProposal:
    # Keep the evidence packet in the interface: a real adapter must reason from facts, not chat history.
    # Stop rather than invent an explanation when the deterministic evaluator rejected the evidence.
    assert assessment.status == "evidence_valid", "Reasoning must stop on an audit failure."
    # Change the scenario coverage instead of blindly searching higher values in the conveyor regime.
    next_spec = ExperimentSpec(
        experiment_id="carrot-seed-opportunity-context-smoke-v1",
        decision_graph_id="carrot_investment",
        parameter_name="seed_opportunity_value",
        baseline_value=20.0,
        # Revisit 40 only as a predeclared contrast under a different deterministic scenario.
        candidate_values=(10.0, 20.0, 40.0),
        suite=Suite(
            opponent_id="future.deterministic.competing-opportunity-scenario",
            seeds=tuple(range(72, 82)),
        ),
        purpose="Test whether a different deterministic opportunity regime changes the seed-option threshold.",
    )
    # Return an explanation and envelope together; neither performs an evaluation or source edit.
    return ReasoningProposal(
        hypothesis=(
            "A static seed opportunity of 20 is supported against the conveyor, "
            "but another decision graph may create states where retaining the option has greater value."
        ),
        explanation=(
            "The completed bracket rules out higher static values for the observed suite, "
            "so the next test changes scenario coverage rather than blindly extending the same range."
        ),
        next_experiment=next_spec,
    )


# This creates a proposal only. Approval and any later run remain outside this cell.
proposal = mock_reasoning_loop(packet, assessment)
# The next cell presents the proposal without exposing the implementation details.

In [26]:
from IPython.display import Markdown, display


def render_reasoning_proposal(proposal: ReasoningProposal) -> None:
    """Presentation only: clearing this output never changes the proposal."""
    spec = proposal.next_experiment
    candidates = ", ".join(f"`{value:g}`" for value in spec.candidate_values)
    display(Markdown(
        f"""### Proposed next bounded test

**Hypothesis:** {proposal.hypothesis}

| Field | Proposed value |
|:---|:---|
| Experiment | `{spec.experiment_id}` |
| Decision graph | `{spec.decision_graph_id}` |
| Parameter | `{spec.parameter_name}`; baseline `{spec.baseline_value:g}` |
| Candidate envelope | {candidates} |
| Scenario | `{spec.suite.opponent_id}` |
| Matched suite | seeds {spec.suite.seeds[0]}–{spec.suite.seeds[-1]}; both seats; {spec.suite.steps} turns; clean audit required |

**Why this test**

{proposal.explanation}

> This is a mock proposal, not an approved experiment. A human must approve this new envelope before a runner may execute it.
"""
    ))


render_reasoning_proposal(proposal)

### Proposed next bounded test

**Hypothesis:** A static seed opportunity of 20 is supported against the conveyor, but another decision graph may create states where retaining the option has greater value.

| Field | Proposed value |
|:---|:---|
| Experiment | `carrot-seed-opportunity-context-smoke-v1` |
| Decision graph | `carrot_investment` |
| Parameter | `seed_opportunity_value`; baseline `20` |
| Candidate envelope | `10`, `20`, `40` |
| Scenario | `future.deterministic.competing-opportunity-scenario` |
| Matched suite | seeds 72–81; both seats; 720 turns; clean audit required |

**Why this test**

The completed bracket rules out higher static values for the observed suite, so the next test changes scenario coverage rather than blindly extending the same range.

> This is a mock proposal, not an approved experiment. A human must approve this new envelope before a runner may execute it.


## 7. The future integration seam: contextual seed opportunity

The carrot graph should continue to estimate its own carrot return. A higher-level planner can eventually provide the value of preserving the seed/tile/action capacity. The following is only a transparent sketch of that interface—not an economic model and not a policy change.

In [13]:
@dataclass(frozen=True)
class FarmContext:
    alternative_crop_value: float
    tile_scarcity_value: float
    farmer_time_pressure: float


def mock_opportunity_value(context: FarmContext) -> float:
    """
    Future planner-to-carrot-graph interface.

    The weights are placeholders. A real graph must provide provenance and
    be calibrated by the same local evaluator—not trusted because it sounds reasonable.
    """
    return (
        context.alternative_crop_value
        + context.tile_scarcity_value
        + context.farmer_time_pressure
    )


quiet_farm = FarmContext(alternative_crop_value=8, tile_scarcity_value=4, farmer_time_pressure=3)
contested_farm = FarmContext(alternative_crop_value=18, tile_scarcity_value=12, farmer_time_pressure=10)

for name, context in [("quiet farm", quiet_farm), ("contested farm", contested_farm)]:
    print(f"{name:15}: proposed opportunity value = {mock_opportunity_value(context):.1f}")

quiet farm     : proposed opportunity value = 15.0
contested farm : proposed opportunity value = 40.0


## 8. What this notebook deliberately does *not* do

- It does not claim that the mock contextual formula is valid.
- It does not call the real evaluator or run a simulator.
- It does not provision Foundry or Azure resources.
- It does not use a model call.
- It does not modify `CarrotParameters` in Kaggriculture source.
- It does not allow the reasoning layer to override failed audits or write a policy change.

## 9. Concrete next implementation step: replace one mock, preserve every boundary

The first real implementation is intentionally small: replace only `mock_inner_loop`. Do **not** build Foundry integration, an autonomous searcher, or a multi-graph planner yet. The goal is to prove that the new project interface can reproduce a result we already trust.

### 9.1 Build one versioned experiment specification

Create a human-readable specification for the completed coarse seed-opportunity bracket: baseline 20; candidate values 10, 40, 60, and 80; deterministic conveyor opponent; seeds 42–71; both seats; 720 turns; and a clean audit requirement. The schema must reject unknown parameter names, duplicate candidate values, an empty suite, a target revision/hash mismatch, or an unbounded candidate request.

The specification is the approval boundary. The runner may execute all values explicitly named in it, but cannot add a fifth value because an earlier result looks interesting.

In [ ]:
{
  "schema_version": 1,
  "experiment_id": "carrot-seed-opportunity-coarse-v1",
  "status": "approved",
  "decision_graph_id": "carrot_investment",

  "target": {
    "expected_revision": "...",
    "expected_policy_sha256": "...",
    "expected_engine_sha256": "..."
  },

  "parameter": {
    "name": "seed_opportunity_value",
    "baseline_value": 20.0,
    "candidate_values": [10.0, 40.0, 60.0, 80.0]
  },

  "suite": {
    "opponent_id": "target.agents.carrot.shared.carrot_conveyor_agent",
    "seeds": [42, 43, "...", 71],
    "seats": [0, 1],
    "steps": 720
  },

  "evaluation": {
    "checkpoint_policy": "before_every_decision",
    "clean_audit_required": true,
    "reuse_no_divergence_baseline": true
  },

  "purpose": "Reproduce the existing coarse static seed-opportunity bracket.",
  "human_decision_required": true
}

### 9.2 Implement a local inner-loop adapter

The adapter validates the specification, binds explicitly to the sibling Kaggriculture checkout, creates or reuses the matching immutable baseline cache, and calls the already-proven checkpoint evaluator for each candidate. It must use the existing rules: replay until first divergence; reuse an exact no-divergence baseline; otherwise restore pre-decision state and run the suffix; then clean-audit every result. It never writes Kaggriculture source.

### 9.3 Persist the evidence packet

Write one immutable JSON report that has the real `EvidencePacket` shape sketched above. It needs target revision and hashes, experiment-spec identity, opponent, seeds/seats/steps, candidate old/new values, per-match outcomes, paired reward statistics, divergence and suffix distributions, timing components, and clean-audit status. Generated caches and reports stay ignored and disposable; specifications and schemas are source-controlled.

### 9.4 Verify reproduction before adding intelligence

The smoke test succeeds only if this new interface reproduces the already-recorded coarse result: value 10 inert; values 40, 60, and 80 harmful in increasing degree; all clean audits passing. We compare outcomes and classifications, not wall-clock timing, because timing is hardware-load sensitive. A mismatch means repair the local runner before extending the system.

### 9.5 Preserve the broader roadmap

This first slice is step one of the larger sequence we do not want to lose:

1. Define the versioned experiment-spec schema and human approval boundary.
2. Build the local deterministic inner-loop runner around the proven evaluator.
3. Emit a durable, structured evidence packet.
4. Add a deterministic local evals layer that classifies evidence and blocks audit failures.
5. Add the Foundry-facing reasoning adapter last, so it proposes the next bounded test from evidence but never becomes the scoring authority or policy editor.

After the reproduction smoke test passes, the next narrow feature is a graph-adapter contract for `carrot_investment`: eligibility, explanation, utility margin, and scenario coverage. Only after that do we add a second decision graph or make `seed_opportunity_value` contextual.